# Physics-Informed Orbital Decay Prediction using Deep Neural Networks
A PyTorch-based deep neural network was developed to predict satellite orbital decay time using orbital and environmental features. The model was evaluated using MAE, RMSE, and R², demonstrating the application of deep learning to an aerospace prediction problem.


In [1]:
# imports libraries
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset, random_split
#from google.colab import files #uncomment if running in Colab to enable file upload
import os, warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  device: {device}')

PyTorch 2.11.0  |  device: cpu


In [4]:
#  Read, Check missing values &  Prepare data
df = pd.read_csv("datasets/orbital_decay.csv")
print(f"Shape: {df.shape}")
display(df.head())

INPUT_COLS = ['initial_altitude_km', 'satellite_mass_kg', 'cross_sectional_area_m2',
              'orbital_eccentricity', 'solar_activity_index', 'drag_coefficient']
X_raw = df[INPUT_COLS].values.astype(np.float32)
y_raw = df['decay_time_days'].values.astype(np.float32)

Shape: (2000, 7)


,initial_altitude_km,satellite_mass_kg,cross_sectional_area_m2,orbital_eccentricity,solar_activity_index,drag_coefficient,decay_time_days
0,349.816048,29.983722,1.113165,0.036600,177.758527,2.926071,47.064455
1,546.470458,106.957216,0.200391,0.035404,73.705209,2.401673,15000.000000
2,272.729987,3.595370,0.074763,0.015212,164.456158,1.775107,69.268416
3,444.741158,115.385267,2.267399,0.014607,135.945132,1.709241,596.504298
4,279.869513,3.972165,0.032477,0.029621,78.361074,2.271352,549.641325


In [5]:
# feature engineering

eps = 1e-8
def engineer_features(X):
    return np.column_stack([
        np.log(X[:, 0] + eps),                               # log altitude
        np.log(X[:, 1] + eps),                               # log mass
        np.log(X[:, 2] + eps),                               # log area
        X[:, 3],                                             # eccentricity 
        np.log(X[:, 4]),                                     # log solar index
        np.log(X[:, 5]),                                     # log drag coefficient
        np.log(X[:, 1] / (X[:, 5] * X[:, 2] + eps) + eps), # log ballistic coefficient
    ])

X_feat = engineer_features(X_raw).astype(np.float32)
y_log  = np.log(y_raw) 

# Compute normalisation stats
FEAT_MEAN = X_feat.mean(axis=0)
FEAT_STD  = X_feat.std(axis=0) + 1e-8
Y_MEAN    = float(y_log.mean())
Y_STD     = float(y_log.std())
print('FEAT_MEAN =', FEAT_MEAN.tolist())
print('FEAT_STD  =', FEAT_STD.tolist())
print(f'Y_MEAN={Y_MEAN:.6f}  Y_STD={Y_STD:.6f}')

X_norm = (X_feat - FEAT_MEAN) / FEAT_STD
y_norm = (y_log  - Y_MEAN)    / Y_STD

# DataLoaders
X_t = torch.tensor(X_norm, dtype=torch.float32)
y_t = torch.tensor(y_norm, dtype=torch.float32).unsqueeze(1)
dataset = TensorDataset(X_t, y_t)
n_val   = int(0.10 * len(dataset))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=64,  shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False)
print(f'Train: {n_train}  Val: {n_val}')

FEAT_MEAN = [5.947067737579346, 3.3882811069488525, -1.0494297742843628, 0.025138908997178078, 4.997615814208984, 0.7919483780860901, 3.6457571983337402]
FEAT_STD  = [0.30201563239097595, 1.753933310508728, 1.782111644744873, 0.014546003192663193, 0.35915258526802063, 0.1935907006263733, 1.1136189699172974]
Y_MEAN=5.901047  Y_STD=1.687059
Train: 1800  Val: 200


In [6]:
#define the network
class DecayPredictionNetwork(nn.Module):
    def __init__(self, in_features=7):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.GELU(),

            nn.Linear(128, 128),
            nn.GELU(),

            nn.Linear(128, 64),
            nn.GELU(),

            nn.Linear(64, 32),
            nn.GELU(),

            nn.Linear(32, 1)
        )

        # Weight initialisation
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.model(x)

model = DecayPredictionNetwork(in_features=7).to(device)
total = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total:,}  ({total*4/1024:.1f} KB as float32)')
print(model)

Parameters: 27,905  (109.0 KB as float32)
DecayPredictionNetwork(
  (model): Sequential(
    (0): Linear(in_features=7, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): GELU(approximate='none')
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): GELU(approximate='none')
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): GELU(approximate='none')
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)


### Performance

The model was trained on 1,800 samples and validated on 200 samples, achieving a best validation MAE of 29.1 days at epoch 465, well below the target threshold of 75 days. The log-space transformation improved prediction accuracy across the full decay range, while the prediction pipeline automated all preprocessing steps.



In [7]:
# Training loop with early stopping
EPOCHS   = 600
PATIENCE = 80

optimiser = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS, eta_min=1e-5)
criterion = nn.HuberLoss(delta=1.0)   
best_val_mae = float('inf')
no_improve   = 0
best_epoch   = 0

for epoch in range(1, EPOCHS + 1):

    #model Training 
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimiser.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        epoch_loss += loss.item()
    scheduler.step()

   #model Validations 
    model.eval()
    preds_v, trues_v = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            preds_v.append(model(xb.to(device)).cpu())
            trues_v.append(yb)

    # Convert normalised log-space back to days 
    pred_days = torch.exp(torch.cat(preds_v) * Y_STD + Y_MEAN)
    true_days = torch.exp(torch.cat(trues_v) * Y_STD + Y_MEAN)
    val_mae   = (pred_days - true_days).abs().mean().item()

   # saving best point 
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_epoch   = epoch
        torch.save(model.state_dict(), 'weights.pkl')
        no_improve = 0
    else:
        no_improve += 1

    if epoch % 50 == 0:
        print(f'Epoch {epoch:4d} | Loss: {epoch_loss/len(train_loader):.4f} | '
              f'Val MAE: {val_mae:.1f}days | Best: {best_val_mae:.1f}days (ep {best_epoch})')

    if no_improve >= PATIENCE:
        print(f'Early stoped at epoch {epoch}')
        break

print(f'\nBest validation Mean Absolute Error(MAE) : {best_val_mae:.1f} days ')
size_kb = os.path.getsize('weights.pkl') / 1024
print(f'weights.pkl  : {size_kb:.1f} KB  ({"File is OK" if size_kb < 1024 else "FILE IS TOO LARGE"})')

Epoch   50 | Loss: 0.0004 | Val MAE: 60.2days | Best: 55.8days (ep 39)
Epoch  100 | Loss: 0.0001 | Val MAE: 47.2days | Best: 44.6days (ep 93)
Epoch  150 | Loss: 0.0002 | Val MAE: 53.5days | Best: 40.1days (ep 137)
Epoch  200 | Loss: 0.0001 | Val MAE: 43.1days | Best: 35.8days (ep 172)
Epoch  250 | Loss: 0.0001 | Val MAE: 35.0days | Best: 31.8days (ep 234)
Epoch  300 | Loss: 0.0001 | Val MAE: 33.4days | Best: 31.2days (ep 268)
Epoch  350 | Loss: 0.0000 | Val MAE: 33.1days | Best: 29.7days (ep 341)
Epoch  400 | Loss: 0.0000 | Val MAE: 30.7days | Best: 29.1days (ep 385)
Epoch  450 | Loss: 0.0000 | Val MAE: 31.4days | Best: 29.1days (ep 385)
Early stoped at epoch 465

Best validation Mean Absolute Error(MAE) : 29.1 days 
weights.pkl  : 112.6 KB  (File is OK)
